# ACT-Break: Activation-guided Contrastive Testing & Suffix Discovery
**Model Target:** `google/gemma-3-1b-it` (or `Qwen/Qwen2.5-0.5B-Instruct`)

ACT-Break is a white-box analysis framework for studying jailbreak direction vectors in language model activation space, optimizing jailbreak suffixes using activation steering directions, and validating model alignment robustness.

### Google Colab Pro Recommendations:
- **GPU Accelerator:** **A100 GPU** or **L4 GPU** is highly recommended for running the `gemma-3-1b-it` optimization steps efficiently.
- **Google Drive Mount:** Highly recommended to preserve outputs, visualization figures, and validation results across runtime recycles.

## 1. Environment Setup & Dependency Installation
This cell will:
1. Mount Google Drive to save results.
2. Clone the ACT-Break repository from GitHub.
3. Install the `uv` package manager and sync project dependencies.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository if not present
import os
if not os.path.exists('/content/ACT-Break'):
    print("Cloning ACT-Break repository...")
    !git clone https://github.com/IrohAmca/ACT-Break.git
    %cd ACT-Break
else:
    print("ACT-Break repository already exists.")
    %cd /content/ACT-Break

# 3. Install uv package manager
import shutil
if not shutil.which("uv"):
    print("Installing uv package manager...")
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]
else:
    print("uv is already installed.")

# 4. Sync dependencies
print("Syncing project dependencies via uv...")
!uv sync

# 5. Disable torch dynamo & force non-interactive matplotlib
os.environ["TORCHDYNAMO_DISABLE"] = "1"
import matplotlib
matplotlib.use("Agg")
print("Environment setup complete!")

## 2. GPU & Hardware Verification
Check the GPU specifications, allocated VRAM, and CUDA version to ensure the runtime is properly configured.

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_mem / (1024**3):.2f} GB")
    print("PyTorch Version:", torch.__version__)
    print("CUDA Version:", torch.version.cuda)
else:
    print("WARNING: Running on CPU. Suffix optimization will be extremely slow!")

## 3. Run Pipeline Steps

You can run the entire pipeline step-by-step. All outputs (activations, probes, figures, validation logs) will be saved in the `outputs/` folder.

### Step 1: Collect Contrastive Activations
Collects compliance vs. refusal activations for AdvBench prompts using the target model (`google/gemma-3-1b-it` or `Qwen/Qwen2.5-0.5B-Instruct`).

In [ ]:
!uv run python -u scripts/01_collect_activations.py

### Step 2: Train Linear Probes
Trains logistic regression probes on target layers to classify compliance vs. refusal activations.

In [ ]:
!uv run python -u scripts/02_train_probe.py

### Step 3: Extract Direction & Visualize
Extracts refusal directions from probe weights, calculates cosine similarity with difference-of-means, and generates PCA/projection plots under `outputs/gemma-3-1b-it/figures/`.

In [ ]:
!uv run python -u scripts/03_extract_direction.py

### Step 4: Activation Steering Validation
Validates if applying steering vectors to the target layers can bypass model safety alignment or cause text generation degradation.

In [ ]:
!uv run python -u scripts/04_steering_validation.py

### Step 5: Multi-Layer GCG Suffix Optimization
Optimizes adversarial jailbreak suffixes using a multi-layer minimax objective.
- **Gradient computation**: Uses the mean gradient across layers L8-L14 to select candidate tokens.
- **Candidate selection**: Evaluates candidate prompts based on the maximum loss (worst-performing layer) across L8-L14, forcing all target layers to converge simultaneously.

In [ ]:
!uv run python -u scripts/05_optimize_suffix.py

### Step 6: Multi-Stage Validation
Performs cross-validation of optimized suffixes, computes text perplexity filters, and tracks hidden state logit lens trajectories.

In [ ]:
!uv run python -u scripts/06_multi_stage_validation.py

## 4. Backup Results to Google Drive
Copies all generated plots, validation reports, and optimized suffixes to Google Drive for persistence.

In [ ]:
import os
import shutil
from datetime import datetime, timezone

drive_dest = "/content/drive/My Drive/ACT-Break-Results"

if os.path.exists("/content/drive/My Drive"):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dest = os.path.join(drive_dest, f"run_{timestamp}")
    os.makedirs(run_dest, exist_ok=True)
    
    print(f"Copying outputs to Google Drive: {run_dest}")
    if os.path.exists("outputs"):
        shutil.copytree("outputs", run_dest, dirs_exist_ok=True)
        print("Successfully copied outputs to Google Drive!")
        
        # Also backup configuration and README
        for file in ["README.md", "config.py"]:
            if os.path.exists(file):
                shutil.copy(file, run_dest)
    else:
        print("No outputs directory found yet. Please run the pipeline steps first.")
else:
    print("Google Drive is not mounted. Please mount Google Drive in the Setup cell first.")